# Sequence-Based Pattern Extraction

This notebook discovers recurring sequences of actions that occur together.

Unlike time-pattern extraction, which answers:

"What time does a device usually turn ON?"

sequence extraction answers:

"What chain of actions usually happens together?"

Example:

son_room:LEAVE
    →
son_room_fan:OFF
    →
son_room_light:OFF

This can be interpreted as a recurring departure routine.

## Step 1

Import required libraries.

We use:

- datetime for timestamp processing
- defaultdict for counting signatures
- statistics for confidence calculations

In [21]:
from collections import defaultdict
from datetime import datetime
import statistics

## Example Dataset

The dataset contains:

- A repeated departure routine
- Random unrelated events

Our goal is to automatically discover that the departure routine repeats.

In [22]:
events = [

    # Day 1 Departure Routine

    ("son_room", "LEAVE", "2026-06-01 08:00"),
    ("son_room_fan", "OFF", "2026-06-01 08:03"),
    ("son_room_light", "OFF", "2026-06-01 08:07"),

    # Day 2 Departure Routine

    ("son_room", "LEAVE", "2026-06-02 08:01"),
    ("son_room_fan", "OFF", "2026-06-02 08:04"),
    ("son_room_light", "OFF", "2026-06-02 08:06"),

    # Day 3 Departure Routine

    ("son_room", "LEAVE", "2026-06-03 07:59"),
    ("son_room_fan", "OFF", "2026-06-03 08:02"),
    ("son_room_light", "OFF", "2026-06-03 08:05"),

    # Day 4 Departure Routine

    ("son_room", "LEAVE", "2026-06-04 08:00"),
    ("son_room_fan", "OFF", "2026-06-04 08:02"),
    ("son_room_light", "OFF", "2026-06-04 08:06"),

    # Random Events

    ("kitchen_light", "ON", "2026-06-01 13:15"),
    ("tv", "ON", "2026-06-01 20:30"),

]

In [ ]:
# parse events

parsed_events = []

for device, action, ts in events:

    parsed_events.append(
        {
            "device": device,
            "action": action,
            "timestamp": datetime.strptime(
                ts,
                "%Y-%m-%d %H:%M"
            )
        }
    )
    

In [37]:
parsed_events

[{'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 0)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 3)},
 {'device': 'son_room_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 7)},
 {'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 1)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 4)},
 {'device': 'son_room_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 6)},
 {'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 3, 7, 59)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 3, 8, 2)},
 {'device': 'son_room_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 3, 8, 5)},
 {'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 4, 8, 0)},
 

## Configuration

MAX_GAP_MINUTES

Maximum gap allowed between consecutive events in a session.

MIN_SEQUENCE_LENGTH

Ignore sessions that are too short.

MAX_SEQUENCE_LENGTH

Prevents extremely long signatures.

MIN_PATTERN_OCCURRENCES

How many times a sequence must repeat before becoming a pattern.

In [24]:
MAX_GAP_MINUTES = 10

MIN_SEQUENCE_LENGTH = 2

MAX_SEQUENCE_LENGTH = 6

MIN_PATTERN_OCCURRENCES = 3

## Step 2

The algorithm first sorts all events chronologically.

Sequence discovery requires an ordered timeline.

In [25]:
ordered = sorted(
    parsed_events,
    key=lambda e: e["timestamp"]
)

ordered

[{'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 0)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 3)},
 {'device': 'son_room_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 7)},
 {'device': 'kitchen_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 1, 13, 15)},
 {'device': 'tv',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 1, 20, 30)},
 {'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 1)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 4)},
 {'device': 'son_room_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 6)},
 {'device': 'son_room',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 3, 7, 59)},
 {'device': 'son_room_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 3, 8, 2)},
 {'devic

## Step 3

Events close together in time are grouped into sessions.

A session represents a burst of related activity.

Example:

08:00 Leave
08:03 Fan OFF
08:07 Light OFF

becomes one session.

In [26]:
def sessionize(events):

    sessions = []

    current = []

    last_ts = None

    for ev in events:

        if (
            last_ts is not None
            and
            (
                ev["timestamp"] - last_ts
            ).total_seconds()
            >
            MAX_GAP_MINUTES * 60
        ):

            if current:
                sessions.append(current)

            current = []

        current.append(ev)

        last_ts = ev["timestamp"]

    if current:
        sessions.append(current)

    return sessions

In [38]:
# create sessions

sessions = sessionize(
    ordered
)

len(sessions)
sessions

[[{'device': 'son_room',
   'action': 'LEAVE',
   'timestamp': datetime.datetime(2026, 6, 1, 8, 0)},
  {'device': 'son_room_fan',
   'action': 'OFF',
   'timestamp': datetime.datetime(2026, 6, 1, 8, 3)},
  {'device': 'son_room_light',
   'action': 'OFF',
   'timestamp': datetime.datetime(2026, 6, 1, 8, 7)}],
 [{'device': 'kitchen_light',
   'action': 'ON',
   'timestamp': datetime.datetime(2026, 6, 1, 13, 15)}],
 [{'device': 'tv',
   'action': 'ON',
   'timestamp': datetime.datetime(2026, 6, 1, 20, 30)}],
 [{'device': 'son_room',
   'action': 'LEAVE',
   'timestamp': datetime.datetime(2026, 6, 2, 8, 1)},
  {'device': 'son_room_fan',
   'action': 'OFF',
   'timestamp': datetime.datetime(2026, 6, 2, 8, 4)},
  {'device': 'son_room_light',
   'action': 'OFF',
   'timestamp': datetime.datetime(2026, 6, 2, 8, 6)}],
 [{'device': 'son_room',
   'action': 'LEAVE',
   'timestamp': datetime.datetime(2026, 6, 3, 7, 59)},
  {'device': 'son_room_fan',
   'action': 'OFF',
   'timestamp': datetime.dat

## Session View

This is one of the most important outputs.

Each session is a candidate routine.

In [28]:
# display session

for i, session in enumerate(sessions):

    print(f"\nSession {i+1}")

    for event in session:

        print(
            event["timestamp"],
            event["device"],
            event["action"]
        )


Session 1
2026-06-01 08:00:00 son_room LEAVE
2026-06-01 08:03:00 son_room_fan OFF
2026-06-01 08:07:00 son_room_light OFF

Session 2
2026-06-01 13:15:00 kitchen_light ON

Session 3
2026-06-01 20:30:00 tv ON

Session 4
2026-06-02 08:01:00 son_room LEAVE
2026-06-02 08:04:00 son_room_fan OFF
2026-06-02 08:06:00 son_room_light OFF

Session 5
2026-06-03 07:59:00 son_room LEAVE
2026-06-03 08:02:00 son_room_fan OFF
2026-06-03 08:05:00 son_room_light OFF

Session 6
2026-06-04 08:00:00 son_room LEAVE
2026-06-04 08:02:00 son_room_fan OFF
2026-06-04 08:06:00 son_room_light OFF


## Step 4

A session is converted into a compact signature.

Example:

son_room:LEAVE
son_room_fan:OFF
son_room_light:OFF

becomes

(
 son_room:LEAVE,
 son_room_fan:OFF,
 son_room_light:OFF
)

In [29]:
def signature(session):

    steps = [

        f"{e['device']}:{e['action']}"

        for e in session
    ]

    return tuple(
        steps[:MAX_SEQUENCE_LENGTH]
    )

In [30]:
# generate signatures
signatures = []

for session in sessions:

    if len(session) < MIN_SEQUENCE_LENGTH:
        continue

    signatures.append(
        signature(session)
    )

signatures

[('son_room:LEAVE', 'son_room_fan:OFF', 'son_room_light:OFF'),
 ('son_room:LEAVE', 'son_room_fan:OFF', 'son_room_light:OFF'),
 ('son_room:LEAVE', 'son_room_fan:OFF', 'son_room_light:OFF'),
 ('son_room:LEAVE', 'son_room_fan:OFF', 'son_room_light:OFF')]

## Step 5

We count how many times each signature appears.

Repeated signatures represent candidate routines.

In [31]:
# count signature occurrences

sig_times = defaultdict(list)

for session in sessions:

    if len(session) < MIN_SEQUENCE_LENGTH:
        continue

    sig = signature(session)

    start = session[0]["timestamp"]

    sig_times[sig].append(
        start.hour * 60
        + start.minute
    )

sig_times

defaultdict(list,
            {('son_room:LEAVE',
              'son_room_fan:OFF',
              'son_room_light:OFF'): [480, 481, 479, 480]})

In [32]:
# confidence helpers

def support_score(count):

    return min(
        count / 10,
        1.0
    )


def consistency_score(
    stddev,
    tolerance=30
):

    return max(
        0,
        1 - stddev / tolerance
    )


def combine(
    support,
    consistency
):

    return (
        support
        +
        consistency
    ) / 2

## Step 6

Patterns are given human-readable descriptions.

Special heuristics help identify common routines.

In [33]:
# human descriptions

def describe(sig):

    has_leave = any(
        step.endswith(":LEAVE")
        for step in sig
    )

    all_off = all(
        step.endswith(":OFF")
        for step in sig[1:]
    )

    if has_leave and all_off:

        return (
            "Departure routine: "
            "home secured / devices switched off"
        )

    return " -> ".join(sig)

In [34]:
# pattern extraction

patterns = []

idx = 1

for sig, starts in sig_times.items():

    if (
        len(starts)
        <
        MIN_PATTERN_OCCURRENCES
    ):
        continue

    mean_start = statistics.mean(
        starts
    )

    stddev = (
        statistics.pstdev(starts)
        if len(starts) > 1
        else 0
    )

    support = support_score(
        len(starts)
    )

    consistency = consistency_score(
        stddev
    )

    confidence = combine(
        support,
        consistency
    )

    patterns.append(
        {
            "pattern_id":
            f"SEQ#{idx:03d}",

            "description":
            describe(sig),

            "steps":
            list(sig),

            "usual_time":
            f"{int(mean_start)//60:02d}:{int(mean_start)%60:02d}",

            "occurrences":
            len(starts),

            "confidence":
            round(confidence,3)
        }
    )

    idx += 1

In [35]:
patterns

[{'pattern_id': 'SEQ#001',
  'description': 'Departure routine: home secured / devices switched off',
  'steps': ['son_room:LEAVE', 'son_room_fan:OFF', 'son_room_light:OFF'],
  'usual_time': '08:00',
  'occurrences': 4,
  'confidence': 0.688}]